In [ ]:
GAMMA = 0.1

In [ ]:
from pathlib import Path
from collections import namedtuple

In [ ]:
Args = namedtuple("args", ["input", "output1", "output2"])
args = Args("2d_gaussian.png", f"Poisson_2d_gaussian_{GAMMA}.pdf", f"Poisson_2d_gaussian_{GAMMA}.png")

In [ ]:
import time

try:
    import numpy as np
except:
    !pip install numpy
    import numpy as np

import scipy.ndimage

try:
    import matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
except:
    !pip install matplotlib
    import matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

try:
    from skimage import io as skimage_io
except:
    !pip install scikit-image
    from skimage import io as skimage_io

try:
    import information_theory as IT
except:
    !pip install "information_theory @ git+https://github.com/vicente-gonzalez-ruiz/information_theory"
    import information_theory as IT

import utils
from PIL import Image

In [ ]:
# apt install cm-super-minimal
# apt install dvipng
plt.rcParams.update({
    "text.usetex": True,
    #"font.family": "Helvetica",
    "font.family": "Serif",
    "text.latex.preamble": r"\usepackage{amsmath} \usepackage{amsfonts}"
})

In [ ]:
try:
    X = skimage_io.imread(args.input)
except FileNotFoundError:
    %run 2d_gaussian.ipynb
    Args = namedtuple("args", ["input", "output1", "output2"])
    args = Args("2d_gaussian.png", f"Poisson_2d_gaussian_{GAMMA}.pdf", f"Poisson_2d_gaussian_{GAMMA}.png")
    X = skimage_io.imread(args.input)

In [ ]:
utils.imshow(X)

In [ ]:
Y = utils.generate_MPGN(X=X, std_dev=0, gamma=GAMMA, poisson_ratio=1.0)
Y = np.clip(a = Y, a_min=0, a_max=255)
PSNR = IT.distortion.PSNR(Y, X)

In [ ]:
utils.imshow(Y)

In [ ]:
string = r"$\hat{\mathbf{s}}=\frac{\mathbf{n}}{"
string += f"{GAMMA}"
string += r"},~\mathbf{n}\sim\mathcal{P}("
string += f"{GAMMA}"
string += r"\mathbf{s})"
string += rf", {PSNR:.2f}"
string += r"~\mathrm{dB}"
string += '$'

plt.title(string)
#plt.imshow(utils.equalize_grayscale_image(Y), cmap="gray")
plt.imshow(Y, cmap="gray")
plt.savefig(args.output1, bbox_inches='tight', pad_inches=0)

In [ ]:
pil_image = Image.fromarray(Y.astype(np.uint8))
pil_image.save(args.output2)